# Dimension Reduction – Python Exercises

This notebook accompanies the lecture slides on **dimension reduction**.  
Work through each exercise in order; the datasets and helper imports are set up for you in the first cell.

**Methods covered:**
1. Principal Component Analysis (PCA)
2. Factor Analysis (FA)
3. Probabilistic PCA (PPCA)
4. Singular Value Decomposition (SVD)
5. Multi-Dimensional Scaling (MDS)
6. Kernel PCA


## Before you start – required libraries

This notebook requires the following Python libraries:

| Library | What it is used for |
|---|---|
| `numpy` | matrix operations, linear algebra |
| `scipy` | eigendecomposition (`scipy.linalg.eigh`) |
| `matplotlib` | plotting |
| `scikit-learn` | PCA, Factor Analysis, Kernel PCA, MDS, datasets |
| `jupyter` / `notebook` | to open and run this notebook |

### Step 1 – check if they are already installed

Run this command in your terminal:

```bash
python3 -c "import numpy, scipy, matplotlib, sklearn, jupyter; print('All good!')"
```

If you see `All good!` you are ready to go.  
If you see a `ModuleNotFoundError`, note the missing library name and go to Step 2.

### Step 2 – install any missing library

For each missing library, run in your terminal:

```bash
python3 -m pip install library_name
```

For example:

```bash
python3 -m pip install scikit-learn
python3 -m pip install numpy
python3 -m pip install scipy
python3 -m pip install matplotlib
python3 -m pip install notebook
```

Or install all of them at once:

```bash
python3 -m pip install numpy scipy matplotlib scikit-learn notebook
```

Once everything is installed, re-run the check in Step 1 to confirm.

## Setup
Run this cell first – it imports all libraries and loads the datasets used throughout the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, make_swiss_roll, load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, FactorAnalysis, KernelPCA
from sklearn.manifold import MDS
from sklearn.metrics import pairwise_distances
from scipy.linalg import eigh
import time

rng = np.random.default_rng(42)

# ── Dataset 1: Iris (small, interpretable) ──────────────────────────────────
iris = load_iris()
X_iris, y_iris = iris.data, iris.target          # (150, 4)

# ── Dataset 2: Digits (medium, classic image dataset) ───────────────────────
digits = load_digits()
X_digits, y_digits = digits.data, digits.target  # (1797, 64)

# ── Dataset 3: Swiss Roll (nonlinear manifold in 3-D) ───────────────────────
X_swiss, t_swiss = make_swiss_roll(n_samples=1000, noise=0.1, random_state=42)

print("Iris shape:   ", X_iris.shape)
print("Digits shape: ", X_digits.shape)
print("Swiss roll:   ", X_swiss.shape)


---
## Exercise 1 – Singular Value Decomposition (SVD)

Any real matrix $\mathbf{X}$ can be written as
$$\mathbf{X} = \mathbf{U}\mathbf{S}\mathbf{V}^\top$$
where $\mathbf{U}$ and $\mathbf{V}$ have orthonormal columns and $\mathbf{S}$ is diagonal with non-negative singular values in decreasing order.

The **rank-$L$ truncated SVD** $\hat{\mathbf{X}}_L = \mathbf{U}_L \mathbf{S}_L \mathbf{V}_L^\top$ is the best rank-$L$ approximation to $\mathbf{X}$ in the Frobenius norm.

### Tasks

**(a)** Compute the full SVD of the centred digits matrix $\mathbf{X}_c$ and plot the singular values.  
**(b)** Reconstruct $\mathbf{X}_c$ using only the top $L \in \{5, 20, 40\}$ singular values/vectors and display one example digit for each.  

In [ ]:
# ── (a) Full SVD and singular values ─────────────────────────────────────────
X_c = X_digits - X_digits.mean(axis=0)

# TODO: compute the full SVD of X_c
# U, s, Vt = np.linalg.svd(X_c, full_matrices=False)

# TODO: plot singular values vs index


In [ ]:
# ── (b) Truncated reconstructions ────────────────────────────────────────────
sample_idx = 0

fig, axes = plt.subplots(1, 4, figsize=(10, 2.5))
axes[0].imshow(X_digits[sample_idx].reshape(8, 8), cmap='gray')
axes[0].set_title('Original'); axes[0].axis('off')

for ax, L in zip(axes[1:], [5, 20, 40]):
    # TODO: reconstruct using top-L components and add back the mean
    # X_hat = U[:, :L] * s[:L] @ Vt[:L, :] + X_digits.mean(axis=0)
    # ax.imshow(X_hat[sample_idx].reshape(8, 8), cmap='gray')
    ax.set_title(f'L = {L}'); ax.axis('off')

plt.tight_layout(); plt.show()


---
## Exercise 2 – Principal Component Analysis (PCA)

PCA finds the directions of **maximum variance** in the data.  
The optimal projection matrix $\hat{\mathbf{W}} = \mathbf{U}_L$ consists of the $L$ eigenvectors of the empirical covariance matrix $\hat{\boldsymbol{\Sigma}}$ with the largest eigenvalues.

### Tasks

**(a)** Centre the Iris data, compute the $4\times4$ covariance matrix, find its eigendecomposition, and project onto the top 2 principal components. Plot coloured by species.  
**(b)** For the digits dataset, plot the scree plot (eigenvalues vs index) and the cumulative fraction of variance explained $F_L = \sum_{j=1}^L \lambda_j / \sum_j \lambda_j$. How many components are needed to explain 90% of the variance?  
**(c)** Apply PCA to Iris without and with standardisation. Plot both 2-D projections side by side and comment.


In [ ]:
# ── (a) From-scratch PCA on Iris ─────────────────────────────────────────────
X_iris_c = X_iris - X_iris.mean(axis=0)
N_iris = X_iris_c.shape[0]

# TODO: compute empirical covariance (divide by N)
# Sigma_hat = X_iris_c.T @ X_iris_c / N_iris

# TODO: eigendecomposition (eigh returns ascending order – reverse it)
# eigvals, eigvecs = np.linalg.eigh(Sigma_hat)

# TODO: project onto top 2 eigenvectors and scatter plot coloured by y_iris


In [ ]:
# ── (b) Scree plot and variance explained ─────────────────────────────────────
X_digits_c = X_digits - X_digits.mean(axis=0)

# TODO: compute eigenvalues of digits covariance matrix
# TODO: plot scree plot (eigenvalues in decreasing order)
# TODO: plot cumulative fraction of variance explained
# TODO: print how many components explain >= 90% of variance


In [ ]:
# ── (d) Covariance vs correlation PCA ─────────────────────────────────────────
X_iris_std = StandardScaler().fit_transform(X_iris)

# TODO: apply PCA (L=2) to X_iris and X_iris_std and plot side by side
# Use sklearn PCA for convenience


---
## Exercise 3 – Factor Analysis (FA)

Factor analysis is a **probabilistic** generalisation of PCA with generative model:
$$p(\boldsymbol{z}) = \mathcal{N}(\boldsymbol{z}\mid\mathbf{0},\mathbf{I}), \qquad
p(\boldsymbol{x}\mid\boldsymbol{z}) = \mathcal{N}(\boldsymbol{x}\mid\mathbf{W}\boldsymbol{z}+\boldsymbol{\mu},\,\boldsymbol{\Psi})$$
where $\boldsymbol{\Psi}$ is **diagonal** (the uniquenesses), giving marginal $p(\boldsymbol{x}) = \mathcal{N}(\boldsymbol{x}\mid\boldsymbol{\mu},\,\mathbf{W}\mathbf{W}^\top+\boldsymbol{\Psi})$.

### Tasks

**(a)** Fit a Factor Analysis model with $L=2$ factors to the standardised Iris data using `sklearn`. Extract the loading matrix $\mathbf{W}$ and uniquenesses $\boldsymbol{\psi}$.  
**(b)** Compute the model covariance $\mathbf{C} = \mathbf{W}\mathbf{W}^\top + \text{diag}(\boldsymbol{\psi})$ and compare it to the sample covariance. Report $\|\mathbf{C} - \mathbf{S}\|_F$.  
**(c)** Obtain the posterior mean embeddings and plot coloured by species.  
**(d)** Vary $L \in \{1,2,3\}$ and report the log-likelihood. What happens as $L$ increases?


In [ ]:
# ── (a) Fit Factor Analysis ───────────────────────────────────────────────────
X_std = StandardScaler().fit_transform(X_iris)

# TODO: fit FactorAnalysis(n_components=2, random_state=42) on X_std
# W = fa.components_.T   # shape (D, L) – the loading matrix
# psi = fa.noise_variance_


In [ ]:
# ── (b) Model covariance vs sample covariance ─────────────────────────────────
# TODO: C_fa = W @ W.T + np.diag(psi)
# TODO: S = np.cov(X_std, rowvar=False)
# TODO: print np.linalg.norm(C_fa - S, 'fro')


In [ ]:
# ── (c) Posterior mean embeddings ─────────────────────────────────────────────
# TODO: Z_fa = fa.transform(X_std)
# TODO: scatter plot coloured by y_iris


In [ ]:
# ── (d) Log-likelihood vs number of factors ───────────────────────────────────
# TODO: for L in [1, 2, 3]:
#     fa_l = FactorAnalysis(n_components=L, random_state=42).fit(X_std)
#     print(f'L={L}  log-likelihood={fa_l.score(X_std):.4f}')
# Comment on what you observe.


---
## Exercise 4 – Probabilistic PCA (PPCA)

PPCA is a special case of FA with $\boldsymbol{\Psi} = \sigma^2 \mathbf{I}$ and orthonormal columns in $\mathbf{W}$.

**Closed-form MLE:**
$$\hat{\mathbf{W}} = \mathbf{U}_L (\mathbf{\Lambda}_L - \hat{\sigma}^2 \mathbf{I})^{1/2}, \qquad
\hat{\sigma}^2 = \frac{1}{D-L}\sum_{j=L+1}^D \lambda_j$$

### Tasks

**(a)** Implement the PPCA MLE from scratch on standardised Iris with $L=2$. Compute $\hat{\mathbf{W}}$ and $\hat{\sigma}^2$.  
**(b)** Compute $\mathbf{C} = \mathbf{W}\mathbf{W}^\top + \hat{\sigma}^2\mathbf{I}$ and compare its Frobenius error to FA from Exercise 3.  
**(c)** Compute the posterior mean $\mathbb{E}[\mathbf{z}\mid\mathbf{x}] = \mathbf{M}^{-1}\mathbf{W}^\top(\mathbf{x}-\bar{\mathbf{x}})$ where $\mathbf{M} = \mathbf{W}^\top\mathbf{W} + \hat{\sigma}^2\mathbf{I}$. Plot coloured by species.  
**(d)** Show numerically that as $\hat{\sigma}^2 \to 0$ the PPCA posterior mean reduces to the standard PCA projection.


In [ ]:
# ── (a) PPCA MLE from scratch ─────────────────────────────────────────────────
X_std = StandardScaler().fit_transform(X_iris)
N, D  = X_std.shape
L     = 2

# TODO: sample covariance S = X_std.T @ X_std / N
# TODO: eigendecomposition (eigh, sort descending)
# TODO: sigma2_hat = mean of the D-L discarded eigenvalues
# TODO: W_hat = U_L @ np.diag(np.sqrt(Lambda_L - sigma2_hat))


In [ ]:
# ── (b) Compare model covariances ─────────────────────────────────────────────
# TODO: C_ppca = W_hat @ W_hat.T + sigma2_hat * np.eye(D)
# TODO: S = np.cov(X_std, rowvar=False)
# TODO: print Frobenius errors for PPCA and FA


In [ ]:
# ── (c) Posterior mean embeddings ─────────────────────────────────────────────
# TODO: M = W_hat.T @ W_hat + sigma2_hat * np.eye(L)
# TODO: Z_ppca = (X_std - X_std.mean(axis=0)) @ W_hat @ np.linalg.inv(M).T
# TODO: scatter plot coloured by y_iris


In [ ]:
# ── (d) Noise-free limit ──────────────────────────────────────────────────────
# TODO: repeat (c) with sigma2 = 1e-8 and compare to standard PCA projection
# PCA projection: Z_pca = X_std @ U_L
# Check np.allclose(Z_ppca_small_noise, Z_pca, atol=1e-4)  (up to sign)


---
## Exercise 5 – Multi-Dimensional Scaling (MDS)

MDS finds low-dimensional embeddings $\{\mathbf{z}_i\}$ that preserve pairwise (dis)similarities.

**Classical MDS** minimises the strain $\|\tilde{\mathbf{K}} - \tilde{\mathbf{Z}}\tilde{\mathbf{Z}}^\top\|_F^2$ where $\tilde{\mathbf{K}} = -\tfrac{1}{2}\mathbf{C}_N\mathbf{D}^{(2)}\mathbf{C}_N$ is the double-centred squared distance matrix.

**Metric MDS** directly minimises $\sum_{i<j}(d_{ij} - \|\mathbf{z}_i-\mathbf{z}_j\|)^2$.

### Tasks

**(a)** Implement classical MDS from scratch on the Iris data: compute $\mathbf{D}^{(2)}$, double-centre to get $\tilde{\mathbf{K}}$, then embed via eigendecomposition. Plot coloured by species.  
**(b)** Show that classical MDS from Euclidean distances equals the PCA scores from Exercise 2(a) up to sign.  
**(c)** Apply metric MDS (`sklearn`) to the Swiss Roll and plot the 2-D result coloured by `t_swiss`. Does it unroll the manifold?  
**(d)** Apply both classical and metric MDS to Iris. Plot side by side and comment on any differences.


In [ ]:
# ── (a) Classical MDS from scratch ────────────────────────────────────────────
N_i = X_iris.shape[0]

# TODO: D2 = pairwise_distances(X_iris, metric='sqeuclidean')
# TODO: C_N = np.eye(N_i) - np.ones((N_i, N_i)) / N_i
# TODO: K_tilde = -0.5 * C_N @ D2 @ C_N
# TODO: eigendecomposition of K_tilde, keep top 2 (eigh, descending)
# TODO: Z_cmds = eigvecs_top2 * np.sqrt(eigvals_top2)
# TODO: scatter plot coloured by y_iris


In [ ]:
# ── (b) Classical MDS = PCA ───────────────────────────────────────────────────
# TODO: align signs of Z_cmds with Z_iris from Exercise 2(a)
# TODO: print np.allclose(Z_cmds_aligned, Z_iris, atol=1e-6)


In [ ]:
# ── (c) Metric MDS on Swiss Roll ──────────────────────────────────────────────
mds = MDS(n_components=2, metric=True, random_state=42, n_jobs=-1)
Z_swiss_mds = mds.fit_transform(X_swiss)

plt.figure(figsize=(6, 5))
plt.scatter(Z_swiss_mds[:, 0], Z_swiss_mds[:, 1],
            c=t_swiss, cmap='viridis', s=10, alpha=0.7)
plt.colorbar(label='t (latent coordinate)')
plt.title('Metric MDS – Swiss Roll')
plt.xlabel('MDS 1'); plt.ylabel('MDS 2')
plt.tight_layout(); plt.show()

# TODO: comment – does metric MDS successfully unroll the manifold?


In [ ]:
# ── (d) Classical vs Metric MDS on Iris ──────────────────────────────────────
mds_metric = MDS(n_components=2, metric=True, random_state=42)
Z_metric = mds_metric.fit_transform(X_iris)

# TODO: plot Z_cmds and Z_metric side by side, coloured by y_iris


---
## Exercise 6 – Comparison of all methods on the Digits dataset

Apply all six methods to the **digits** dataset and produce a $2 \times 3$ grid of 2-D embedding plots, each coloured by digit label (0–9).

**Methods:**
1. PCA  
2. Factor Analysis (FA)  
3. Probabilistic PCA (PPCA)  
4. Classical MDS  
5. Metric MDS  
6. Kernel PCA with RBF kernel  

Discuss:
- Which methods produce the clearest cluster separation?
- How do the running times compare?
- What are the trade-offs between interpretability, computational cost, and embedding quality?


In [ ]:
# ── Exercise 6: comparison of all methods on Digits ──────────────────────────
X_dig_std = StandardScaler().fit_transform(X_digits)

# TODO: fit each method and record 2-D embedding + time
# methods = {}
# t0 = time.time(); methods['PCA']          = PCA(n_components=2).fit_transform(X_dig_std);            print(f"PCA:          {time.time()-t0:.2f}s")
# t0 = time.time(); methods['FA']           = FactorAnalysis(n_components=2, random_state=42).fit_transform(X_dig_std); print(f"FA:           {time.time()-t0:.2f}s")
# t0 = time.time(); ... PPCA from scratch ...
# t0 = time.time(); ... Classical MDS from scratch (on X_dig_std) ...
# t0 = time.time(); methods['Metric MDS']   = MDS(n_components=2, metric=True, random_state=42).fit_transform(X_dig_std); print(f"Metric MDS:   {time.time()-t0:.2f}s")
# t0 = time.time(); methods['Kernel PCA']   = KernelPCA(n_components=2, kernel='rbf', gamma=0.01, random_state=42).fit_transform(X_dig_std); print(f"Kernel PCA:   {time.time()-t0:.2f}s")

# TODO: 2×3 grid of scatter plots coloured by y_digits
# fig, axes = plt.subplots(2, 3, figsize=(15, 9))
# for ax, (name, Z) in zip(axes.ravel(), methods.items()):
#     ax.scatter(Z[:, 0], Z[:, 1], c=y_digits, cmap='tab10', s=5, alpha=0.6)
#     ax.set_title(name); ax.set_xticks([]); ax.set_yticks([])
# plt.suptitle('Dimension reduction – Digits (2-D)', fontsize=14)
# plt.tight_layout(); plt.show()

# TODO: comment on cluster separation, running times, and trade-offs
